# Supa Strikers: broadcast ↔ tactical sync from the on-screen match clock

Links the **broadcast** video (164,488 frames, its own timeline, with cuts, replays and zooms) to the
**tactical** video (199,319 frames, the submission reference) by reading the match clock burned into
the top-left of both feeds. Everything is expressed relative to the **kickoff**, not the start of
each video.

**What it produces** in `outputs/sync/` (gitignored):

| file | content |
|---|---|
| `sync_<mode>.json` | per-source clock segments, offsets, kickoff frames, per-half pair offsets, per-frame broadcast→tactical map with shot labels, tactical→broadcast index, all checks |
| `sync_<mode>_broadcast_frames.csv` | the per-frame broadcast table, for pandas |
| `sync_<mode>_offsets.png`, `sync_<mode>_labels.png`, `sync_<mode>_ocr_audit_*.png` | diagnostics |

**How it works**
1. **OCR the clock** in a fixed crop of each video every `SAMPLE_STEP` frames (12 at 25 fps, chosen
   *not* to divide the fps so that sample phases sweep through every frame of a second).
2. **Fit offsets.** While play is live, `match_clock = video_time − offset`. Each reading `c` at time `t`
   says `offset ∈ (t−c−1, t−c]`; intersecting those intervals by robust voting gives the offset to
   about one frame (40 ms), with no per-frame OCR. Jumps in the offset split the video into segments
   (halftime, edits); frozen clocks (pre-match 00:00, a 45:00 hold) are detected and ignored.
3. **Halves and kickoffs.** A segment that starts near 45:00 after an offset jump is the 2nd half. The
   kickoff frame of each half in each video is `offset + half_start_clock`. The tactical one is checked
   against DATA.md / FAQ.md (11,185 and 113,742).
4. **Label broadcast frames** as `LIVE_WIDE`, `LIVE_CLOSE`, `REPLAY`, `OFF_MATCH` or `UNKNOWN` from: clock
   visible and consistent, the provided broadcast homography `valid` flag, a slow-motion score, and an
   **ATD cross-check** (broadcast ATD players vs tactical ATD players at the mapped frame: live frames
   agree, replays do not).
5. **Refine** the broadcast→tactical offset per half by ±2 s using the same ATD cross-check, because the
   two clock graphics come from different production systems.

**OCR model, recommended: RapidOCR** (PaddleOCR PP-OCR recognition models exported to ONNX, run by
`onnxruntime`). It's CPU-only, ~20 ms per crop, pip-installable on Python 3.13 / Windows, needs no
torch or paddle, and reads both clocks at confidence ≥ 0.999 on the sample screenshots. It runs in
recognition-only mode (`use_det=False`) because the crop is fixed. Alternatives: `easyocr` (needs
torch) and `tesseract` (needs the system binary).
Only **local** OCR is used: the T&C forbid sending videos or extracts to hosted LLM/vision APIs.

**On Kaggle** (detected automatically; the same notebook also runs locally from the repo)
* **Inputs** (Add Input → your private datasets): the tactical dataset (e.g. `bar-ath-tactical`) and a
  broadcast dataset with `BAR-ATH_broadcast.mp4`, `_atd.txt`, `_homography.csv`. Files are found
  anywhere under `/kaggle/input` by their original names, so folder layout doesn't matter. Only the two
  `.mp4` files are required; the ATD and homography files enable the replay check and the refinement.
* **Settings:** Accelerator *None* (CPU is enough: OCR is ~20 ms per crop, decoding is the cost),
  Internet *on* for `pip install rapidocr onnxruntime`. With Internet off, add a dataset containing a
  `wheels/` folder (`pip download rapidocr onnxruntime tqdm -d wheels`).
* **Outputs** go to `/kaggle/working/sync/` and appear in the notebook's Output tab.
* **Resume:** add a previous version's output as an input. Its `sync/cache/` (OCR samples, parsed ATD)
  is copied in, so a re-run skips the video scan.
* `games.py` isn't needed on Kaggle. Its constants (frame counts, pitch, kickoffs, half boundaries from
  DATA.md) are built in as fallbacks.
* **Procedure:** leave `RUN_MODE = "smoke"` → *Run All* (~5–10 min) → check the ROI preview (§9) and
  the checks (§14) → set `RUN_MODE = "full"` → *Save Version → Save & Run All* (~45–75 min on
  Kaggle's 4 vCPUs). Progress bars update every 15 s in committed runs to keep the log short.

**How to run.** `RUN_MODE = "selftest"` (no match data: synthetic videos, ~1 min) → `"smoke"` (coarse
whole-match scan + dense 3 min after kickoff, ~5 min) → `"full"` (every 12th frame of both videos,
~30–45 min on a laptop CPU). Results are cached in `outputs/sync/cache/`, so re-running the analysis
cells is instant. The environment variables `SYNC_RUN_MODE` / `SYNC_OCR` override the config, e.g.
`SYNC_RUN_MODE=full jupyter nbconvert --execute --to notebook notebooks/broadcast_tactical_sync.ipynb`.

**Compliance (RULES.md).** No hand labels anywhere: every decision comes from OCR, the provided
homography and ATD, and fixed thresholds. The thresholds are not tuned on ground truth.

## 1. Config: everything you might change lives here

In [ ]:
import os, sys, json, re, math, time, subprocess, datetime
from pathlib import Path

RUN_MODE    = os.environ.get("SYNC_RUN_MODE", "smoke")     # "selftest" | "smoke" | "full"
OCR_BACKEND = os.environ.get("SYNC_OCR", "rapidocr")       # "rapidocr" (recommended) | "easyocr" | "tesseract"
AUTO_INSTALL = True                  # pip-install tqdm / the OCR backend if missing

GAME = "BAR-ATH"

# Clock crop (x, y, w, h) in a 1920x1080 reference frame; scaled to the real video size.
# Measured on the two sample frames: broadcast "01:19" (dark on white), tactical "28:40" (white on grey).
REF_SIZE = (1920, 1080)
# ---- Clock bounding box: FIXED position, read exactly as given -------------------------------------
# (x, y, w, h) in pixels of a 1920x1080 frame (scaled automatically if a video has another size).
# x, y = top-left corner of the box. Read the numbers off the pixel-grid image in section 9.
# On Kaggle you can override without editing: env SYNC_ROI_TACTICAL / SYNC_ROI_BROADCAST = "x,y,w,h".
CLOCK_ROI_REF = {
    "tactical":  (136, 2, 98, 44),     # verified: reads 03:09, 19:06, ... at >= 0.93
    "broadcast": (95, 110, 90, 52),    # estimate (the scoreboard is lower than in the screenshot): check the grid
}
ROI_MODE          = "fixed"           # "fixed" (default): only the boxes above | "auto": text detection if a box fails
SHOW_ROI_GRID     = True              # section 9: top-left of 3 frames per video with a pixel grid and the box in red
ROI_GRID_REF      = (0, 0, 640, 320)  # area shown in that grid (1920x1080 reference)
ROI_CHECK_PROBES  = 24                # frames across the video used to check each box
ROI_MIN_RATE      = 0.30              # smoke/full stop before the long scan if a box reads a clock on fewer probes
# used only when ROI_MODE == "auto"
ROI_SEARCH_REF    = (0, 0, 700, 300)
AUTO_ROI_MIN_HITS = 4

# OCR sampling
SAMPLE_STEP   = None        # frames between clock reads; None = auto (12 at 25 fps, never a divisor of fps)
MIN_OCR_SCORE = 0.995       # only "confidence 1.00" readings are used (RapidOCR shows 2 decimals: >= 0.995 prints as 1.00);
                            # everything below counts as no reading. Lower it (e.g. 0.9) only if too few readings survive.
BLANK_STD     = 6.0         # crop grey-level std below this = no clock graphic, OCR skipped
MAX_CLOCK_MIN = 130

# Offset fitting
HALF_START_CLOCK      = {1: 0, 2: 45 * 60}   # what the clock shows at each kickoff
HALF_START_TOL_SEC    = 120     # a new half's first reading must be this close to its start clock
MIN_HALFTIME_JUMP_SEC = 15      # ... and the offset must jump by at least this much
OFFSET_TOL_SEC        = 1.5     # offsets closer than this belong to the same segment
MIN_RUN_SAMPLES       = 8       # a segment needs at least this many consistent readings
FROZEN_SPAN_SEC       = 1.1     # one clock value seen for longer than this = frozen clock, ignored

# Smoke mode
SMOKE_COARSE_STEP_SEC = 20      # coarse whole-match scan by seeking
SMOKE_DENSE_SEC       = 180     # dense window after the 1st-half kickoff (per-frame output only here)

# Broadcast shot labels
USE_HOMOGRAPHY   = True         # provided broadcast homography `valid` -> wide vs close
USE_ATD          = True         # provided ATD cross-check (live vs replay) + offset refinement
ATD_MIN_PLAYERS  = 6
ATD_LIVE_M       = 2.0          # median nearest-player distance below this = same moment (live)
ATD_REPLAY_M     = 4.0          # above this = different moment (replay)
ATD_REFINE_MAX_SHIFT = 50       # frames searched each way when refining the pair offset
ATD_REFINE_SAMPLES   = 400
SLOWMO_REL       = 0.35         # motion below this fraction of live motion = slow motion
SLOWMO_SHARE     = 0.40         # share of slow-mo samples that makes a clock-less stretch a replay
ATD_REPLAY_SHARE = 0.30         # share of ATD "different moment" votes that makes it a replay
SHORT_GAP_SEC    = 4.0          # clock-less stretch this short with no replay evidence = live close-up

# Reference values from the repo docs (DATA.md, FAQ.md): tactical kickoffs in video frames
EXPECTED_TACTICAL_KICKOFF = {1: 11185, 2: 113742}
KICKOFF_TOL_FRAMES = 25

PARALLEL     = True     # scan both videos at once (decode and OCR release the GIL)
REUSE_CACHE  = True     # reuse cached OCR samples if the same video / roi / step was scanned before
WRITE_PER_FRAME      = True
WRITE_TACTICAL_INDEX = True
SEED = 0

for _s in ("tactical", "broadcast"):
    _env = os.environ.get(f"SYNC_ROI_{_s.upper()}")
    if _env:
        CLOCK_ROI_REF[_s] = tuple(int(float(v)) for v in _env.split(","))
        print(f"CLOCK_ROI_REF['{_s}'] = {CLOCK_ROI_REF[_s]} (from SYNC_ROI_{_s.upper()})")

# Kaggle (detected automatically; the same notebook runs locally from the repo)
KAGGLE_INPUT    = Path("/kaggle/input")
KAGGLE_WORKING  = Path("/kaggle/working")
ON_KAGGLE       = KAGGLE_INPUT.exists() or "KAGGLE_KERNEL_RUN_TYPE" in os.environ
KAGGLE_BATCH    = os.environ.get("KAGGLE_KERNEL_RUN_TYPE") == "Batch"     # "Save & Run All" commit
DATA_SEARCH_DIRS = []      # extra folders to search for BAR-ATH_* files; /kaggle/input is always searched
OFFLINE_WHEELS_GLOB = "/kaggle/input/*/wheels"   # Internet off: a dataset folder with rapidocr/onnxruntime/tqdm wheels
RESUME_CACHE_GLOB   = "/kaggle/input/*/sync/cache"  # a previous run's output added as input -> its OCR cache is reused

In [ ]:
# ---- paths: games.py is the single source of file names (data/README.md) ----
def find_repo_root():
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "games.py").exists():
            return p
    return Path.cwd()

REPO = find_repo_root()
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
try:
    import games
    HAVE_GAMES = True
except ImportError:
    games, HAVE_GAMES = None, False

_SUFFIX = {"video": ".mp4", "atd_raw": "_atd.txt", "homography": "_homography.csv"}

ASSET_OVERRIDE = {}
_FOUND = {}

def _search_roots():
    roots = [Path(p) for p in DATA_SEARCH_DIRS]
    if os.environ.get("MTAH_DATA_DIR"):
        roots.append(Path(os.environ["MTAH_DATA_DIR"]))
    if KAGGLE_INPUT.exists():
        roots.append(KAGGLE_INPUT)
    roots.append(REPO / "data")
    return [r for r in roots if r.exists()]

def asset(source, name):
    """Path of a provided file. Order: override -> games.py (repo) -> search of Kaggle inputs / data dirs.
    On Kaggle the file can sit anywhere under /kaggle/input as long as it keeps its original name."""
    if (source, name) in ASSET_OVERRIDE:
        return Path(ASSET_OVERRIDE[(source, name)])
    fname = f"{GAME}_{source}{_SUFFIX[name]}"
    if HAVE_GAMES:
        p = Path(games.asset_paths(GAME, source=source)[name])
        if p.exists():
            return p
    if fname not in _FOUND:
        hit = next((h for r in _search_roots() for h in r.rglob(fname)), None)
        _FOUND[fname] = hit
    return _FOUND[fname] or (REPO / "data" / source / fname)

EXPECTED_FRAMES = ({s: games.SOURCES[s]["n_video_frames"] for s in ("tactical", "broadcast")}
                   if HAVE_GAMES else {"tactical": 199319, "broadcast": 164488})
PITCH_M = tuple(games.GAMES[GAME]["pitch"]) if HAVE_GAMES else (105.3, 68.0)
# graded halves in tactical video frames (DATA.md "Which frames are graded"); used for the end of each half
HALF_VIDEO_BLOCKS = (games.half_video_blocks(GAME) if HAVE_GAMES else [(1, 11185, 90651), (2, 110808, 190197)])
SOURCES = ("tactical", "broadcast")
OUT_DIR = (KAGGLE_WORKING if ON_KAGGLE and KAGGLE_WORKING.exists() else REPO / "outputs") / "sync"
TQDM_MIN = 15.0 if KAGGLE_BATCH else 1.0      # fewer progress lines in committed Kaggle logs
VIDEO = {s: asset(s, "video") for s in SOURCES}

if RUN_MODE == "selftest":
    # Synthetic videos with known offsets, a replay, a close-up, frozen clocks and a short 'half'.
    OUT_DIR = OUT_DIR / "selftest"
    VIDEO = {s: OUT_DIR / f"synthetic_{s}.mp4" for s in SOURCES}
    HALF_START_CLOCK = {1: 0, 2: 40}
    HALF_START_TOL_SEC, MIN_HALFTIME_JUMP_SEC, MIN_RUN_SAMPLES = 10, 5, 6
    ASSET_OVERRIDE = {(s, n): OUT_DIR / f"synthetic_{s}{_SUFFIX[n]}" for s in SOURCES for n in ("atd_raw", "homography")}
    EXPECTED_TACTICAL_KICKOFF = {1: 209, 2: 1555}
    KICKOFF_TOL_FRAMES = 1
    REUSE_CACHE = False
    EXPECTED_FRAMES = {"tactical": 2500, "broadcast": 2125}

assert RUN_MODE in ("selftest", "smoke", "full"), RUN_MODE
CACHE_DIR = OUT_DIR / "cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
# resume: copy a previous run's OCR / ATD cache (Kaggle: add that run's output as an input)
import glob, shutil
for d in glob.glob(RESUME_CACHE_GLOB):
    n = 0
    for f in Path(d).glob("*"):
        if f.is_file() and not (CACHE_DIR / f.name).exists():
            shutil.copy2(f, CACHE_DIR / f.name); n += 1
    if n:
        print(f"resume: copied {n} cache file(s) from {d}")

print(f"RUN_MODE={RUN_MODE}  OCR={OCR_BACKEND}  kaggle={ON_KAGGLE}{' (batch)' if KAGGLE_BATCH else ''}  "
      f"repo={REPO}  games.py={'yes' if HAVE_GAMES else 'no'}  out={OUT_DIR}")
for s in SOURCES:
    print(f"  {s:9s} video: {VIDEO[s]}  exists={VIDEO[s].exists() or RUN_MODE == 'selftest'}")
    for n in ("atd_raw", "homography"):
        if RUN_MODE != "selftest" and not (s == "tactical" and n == "homography"):
            print(f"  {s:9s} {n:10s}: {asset(s, n)}  exists={asset(s, n).exists()}")
if RUN_MODE != "selftest" and not all(VIDEO[s].exists() for s in SOURCES):
    raise FileNotFoundError(
        "video(s) not found. Locally: put the bundle in data/ (DATA.md). On Kaggle: Add Input -> the dataset(s) "
        "with BAR-ATH_tactical.mp4 and BAR-ATH_broadcast.mp4 (any folder, original file names), or set DATA_SEARCH_DIRS.")

## 2. Install and import

In [ ]:
def _pip(*pkgs):
    """pip into this kernel's interpreter. Kaggle with Internet off: install from a wheels dataset.
    uv-made venvs have no pip, so fall back to `uv pip`."""
    print("installing", *pkgs)
    wheels = glob.glob(OFFLINE_WHEELS_GLOB)
    extra = ["--no-index", "--find-links", wheels[0]] if wheels else []
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *extra, *pkgs], check=False)
    if r.returncode != 0 and not ON_KAGGLE:
        subprocess.run(["uv", "pip", "install", "--python", sys.executable, *pkgs], check=False)
    elif r.returncode != 0:
        print("pip failed. On Kaggle turn Internet ON (Settings), or add a dataset with the wheels "
              "(pip download rapidocr onnxruntime tqdm -d wheels) and keep OFFLINE_WHEELS_GLOB pointing at it.")

_need = []
try:
    import tqdm  # noqa: F401
except ImportError:
    _need.append("tqdm")
if OCR_BACKEND == "rapidocr":
    try:
        import rapidocr  # noqa: F401
    except ImportError:
        try:
            import rapidocr_onnxruntime  # noqa: F401
        except ImportError:
            _need += ["rapidocr", "onnxruntime"]
elif OCR_BACKEND == "easyocr":
    try:
        import easyocr  # noqa: F401
    except ImportError:
        _need.append("easyocr")
elif OCR_BACKEND == "tesseract":
    try:
        import pytesseract  # noqa: F401
    except ImportError:
        _need.append("pytesseract")     # plus the tesseract binary on PATH
if _need and AUTO_INSTALL and not os.environ.get("SS_SKIP_INSTALL"):
    _pip(*_need)
    import importlib; importlib.invalidate_caches()

import numpy as np
import pandas as pd
import cv2
import matplotlib
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from concurrent.futures import ThreadPoolExecutor

cv2.setNumThreads(max(1, (os.cpu_count() or 2) // 2))
print("numpy", np.__version__, "| pandas", pd.__version__, "| opencv", cv2.__version__)

## 3. Clock OCR

`parse_clock` turns recogniser text into seconds and fixes the usual confusions (`O→0`, `l→1`, `S→5`…).
`ClockOCR.read` skips blank crops, upscales ×3 and runs the backend in recognition-only mode.

In [ ]:
_CLOCK_RE = re.compile(r"(\d{1,3})\s*[:.;,]\s*(\d{2})")
_FIX = str.maketrans({"O": "0", "o": "0", "D": "0", "Q": "0", "I": "1", "l": "1", "|": "1", "i": "1",
                      "!": "1", "S": "5", "s": "5", "B": "8", "Z": "2", "z": "2", "G": "6", "g": "9", "q": "9"})

def parse_clock(text):
    """'45:12' -> 2712 seconds. None if the text is not a plausible clock."""
    if not text:
        return None
    t = text.translate(_FIX)
    m = _CLOCK_RE.search(t)
    if m:
        mm, ss = int(m.group(1)), int(m.group(2))
    else:
        d = re.sub(r"\D", "", t)
        if len(d) != 4:
            return None
        mm, ss = int(d[:2]), int(d[2:])
    if ss > 59 or mm > MAX_CLOCK_MIN:
        return None
    return mm * 60 + ss


class ClockOCR:
    """One instance per thread. Recognition only: the crop is fixed, no text detection needed."""

    def __init__(self, backend=OCR_BACKEND):
        self.backend = backend
        if backend == "rapidocr":
            try:
                from rapidocr import RapidOCR
                self._eng, self._api = RapidOCR(), "v3"
            except ImportError:
                from rapidocr_onnxruntime import RapidOCR
                self._eng, self._api = RapidOCR(), "v1"
        elif backend == "easyocr":
            import easyocr
            try:
                import torch
                gpu = torch.cuda.is_available()
            except ImportError:
                gpu = False
            self._eng = easyocr.Reader(["en"], gpu=gpu, verbose=False)
        elif backend == "tesseract":
            import pytesseract
            self._eng = pytesseract
        else:
            raise ValueError(backend)

    @staticmethod
    def _prep(crop):
        up = cv2.resize(crop, None, fx=3, fy=3, interpolation=cv2.INTER_CUBIC)
        return cv2.copyMakeBorder(up, 10, 10, 10, 10, cv2.BORDER_REPLICATE)

    def read(self, crop):
        """(text, score). Empty text and score 0 for a blank crop."""
        grey = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)
        if float(grey.std()) < BLANK_STD:
            return "", 0.0
        img = self._prep(crop)
        if self.backend == "rapidocr":
            if self._api == "v3":
                r = self._eng(img, use_det=False, use_cls=False, use_rec=True)
                txts, scores = list(r.txts or ()), list(r.scores or ())
            else:
                res, _ = self._eng(img, use_det=False, use_cls=False, use_rec=True)
                res = res or []
                txts, scores = [x[0] for x in res], [float(x[1]) for x in res]
            return "".join(txts).strip(), float(min(scores)) if scores else 0.0
        if self.backend == "easyocr":
            out = sorted(self._eng.readtext(img, allowlist="0123456789:", detail=1, paragraph=False),
                         key=lambda o: o[0][0][0])
            return "".join(o[1] for o in out).strip(), float(min((o[2] for o in out), default=0.0))
        g = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        _, b = cv2.threshold(g, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        if b.mean() < 127:
            b = 255 - b                      # tesseract wants dark text on white
        text = self._eng.image_to_string(b, config="--psm 7 -c tessedit_char_whitelist=0123456789:")
        return text.strip(), 1.0 if text.strip() else 0.0

    def detect(self, img):
        """Text detection + recognition on a larger region: [(x0, y0, x1, y1, text, score)]."""
        out = []
        if self.backend == "rapidocr":
            if self._api == "v3":
                r = self._eng(img, use_det=True, use_cls=False, use_rec=True)
                boxes = [] if r.boxes is None else list(r.boxes)
                items = zip(boxes, list(r.txts or ()), list(r.scores or ()))
            else:
                res, _ = self._eng(img, use_det=True, use_cls=False, use_rec=True)
                items = [(x[0], x[1], float(x[2])) for x in (res or [])]
        elif self.backend == "easyocr":
            items = [(o[0], o[1], o[2]) for o in self._eng.readtext(img, detail=1, paragraph=False)]
        else:
            items = []                      # tesseract: no auto-locate, use CLOCK_ROI_REF
        for box, text, score in items:
            b = np.asarray(box, dtype=np.float64).reshape(-1, 2)
            out.append((b[:, 0].min(), b[:, 1].min(), b[:, 0].max(), b[:, 1].max(), str(text), float(score)))
        return out


assert parse_clock("01:19") == 79 and parse_clock("45:00") == 2700 and parse_clock("9O:l2") == 5412
assert parse_clock("0119") == 79 and parse_clock("12:75") is None and parse_clock("") is None
_t0 = time.time(); _ocr_probe = ClockOCR(); print(f"OCR backend ready: {OCR_BACKEND} ({time.time() - _t0:.1f}s)")

## 4. Video scanning

`scan_video` reads the clock crop on the requested frames, either sequentially (`grab()` every frame,
`retrieve()` only the wanted ones: exact and fast) or by seeking (smoke mode's coarse scan). For the
broadcast it also measures motion between each sampled frame and the next (the slow-motion signal).

In [ ]:
def video_info(path):
    cap = cv2.VideoCapture(str(path))
    if not cap.isOpened():
        raise FileNotFoundError(f"cannot open {path}. Is the match data in data/ (see DATA.md)?")
    info = {"fps": float(cap.get(cv2.CAP_PROP_FPS)), "n_frames": int(cap.get(cv2.CAP_PROP_FRAME_COUNT)),
            "width": int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), "height": int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))}
    cap.release()
    return info


def roi_px(source, info):
    x, y, w, h = CLOCK_ROI_REF[source]
    sx, sy = info["width"] / REF_SIZE[0], info["height"] / REF_SIZE[1]
    return (int(round(x * sx)), int(round(y * sy)), max(8, int(round(w * sx))), max(8, int(round(h * sy))))


def auto_step(fps):
    """About half a second, and never a divisor of round(fps), so sample phases sweep every frame."""
    s = max(1, int(round(fps / 2.0)))
    while s > 1 and int(round(fps)) % s == 0:
        s -= 1
    return s


_STRICT_CLOCK_RE = re.compile(r"\d{1,3}\s*[:;]\s*\d{2}")

def locate_clock_roi(source, info, frames, desc=""):
    """Find the clock automatically: OCR text detection over ROI_SEARCH_REF on the probe frames, keep boxes
    reading MM:SS (trimmed to the clock characters when merged with a team name), median position + padding."""
    sx, sy = info["width"] / REF_SIZE[0], info["height"] / REF_SIZE[1]
    X, Y, W, H = (int(round(v * k)) for v, k in zip(ROI_SEARCH_REF, (sx, sy, sx, sy)))
    ocr = ClockOCR()
    boxes = []
    cap = cv2.VideoCapture(str(VIDEO[source]))
    for f in tqdm(frames, desc=desc or f"locate clock {source}", unit="seek", mininterval=TQDM_MIN):
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(f)); ok, frame = cap.read()
        if not ok:
            continue
        for x0, y0, x1, y1, text, score in ocr.detect(frame[Y:Y + H, X:X + W]):
            t = text.translate(_FIX)
            m = _STRICT_CLOCK_RE.search(t)
            if not m or score < MIN_OCR_SCORE or parse_clock(m.group(0)) is None:
                continue
            L = max(len(t), 1)
            a, b = x0 + (x1 - x0) * m.start() / L, x0 + (x1 - x0) * m.end() / L
            boxes.append((a + X, y0 + Y, b + X, y1 + Y))
    cap.release()
    if len(boxes) < AUTO_ROI_MIN_HITS:
        return None, len(boxes)
    B = np.asarray(boxes)
    med = np.median(B, axis=0)
    centre = np.c_[(B[:, 0] + B[:, 2]) / 2, (B[:, 1] + B[:, 3]) / 2]
    near = np.hypot(*(centre - [(med[0] + med[2]) / 2, (med[1] + med[3]) / 2]).T) < 0.5 * (med[3] - med[1]) + 20
    if near.sum() >= AUTO_ROI_MIN_HITS:            # drop clock-like text elsewhere (ads, other graphics)
        med = np.median(B[near], axis=0)
    pad = 0.2 * (med[3] - med[1])
    x0, y0 = max(0, int(med[0] - pad)), max(0, int(med[1] - pad))
    x1, y1 = min(info["width"], int(med[2] + pad)), min(info["height"], int(med[3] + pad))
    return (x0, y0, x1 - x0, y1 - y0), int(near.sum())


def _thumb(frame):
    return cv2.resize(cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY), (64, 36), interpolation=cv2.INTER_AREA).astype(np.float32)


def scan_video(path, frames, roi, motion=False, mode="seq", desc="", position=0, keep_crops=0):
    """DataFrame(frame, text, score, clock, motion) for the requested frames, plus a few crops for audit."""
    frames = np.unique(np.asarray(frames, dtype=np.int64))
    ocr = ClockOCR()
    x, y, w, h = roi
    rows, crops = [], []
    rng = np.random.default_rng(SEED)
    keep = set(rng.choice(frames, size=min(keep_crops, len(frames)), replace=False).tolist()) if keep_crops else set()

    def handle(f, frame, nxt):
        crop = frame[y:y + h, x:x + w]
        text, score = ocr.read(crop)
        clk = parse_clock(text) if score >= MIN_OCR_SCORE else None
        mot = float(np.abs(_thumb(frame) - _thumb(nxt)).mean()) if (motion and nxt is not None) else np.nan
        rows.append((int(f), text, score, np.nan if clk is None else float(clk), mot))
        if f in keep:
            crops.append((int(f), crop.copy(), text))

    cap = cv2.VideoCapture(str(path))
    if mode == "seek":
        bad_seeks = 0
        for f in tqdm(frames, desc=desc, position=position, unit="seek", mininterval=TQDM_MIN):
            cap.set(cv2.CAP_PROP_POS_FRAMES, int(f))
            if int(round(cap.get(cv2.CAP_PROP_POS_FRAMES))) != f:
                bad_seeks += 1
            ok, frame = cap.read()
            if not ok:
                continue
            nxt = None
            if motion:
                ok2, nxt = cap.read()
                nxt = nxt if ok2 else None
            handle(f, frame, nxt)
        if bad_seeks:
            print(f"[{desc}] WARNING: {bad_seeks} inaccurate seeks; coarse smoke readings may be off by a few frames")
    else:
        start, stop = int(frames[0]), int(frames[-1]) + (1 if motion else 0)
        pos = 0
        if start > 0:
            cap.set(cv2.CAP_PROP_POS_FRAMES, start)
            pos = int(round(cap.get(cv2.CAP_PROP_POS_FRAMES)))
            if pos != start:                 # inexact seek: fall back to reading from frame 0
                cap.release(); cap = cv2.VideoCapture(str(path)); pos = 0
        want = set(frames.tolist())
        pending = None
        bar = tqdm(total=stop - pos + 1, desc=desc, position=position, unit="fr", mininterval=TQDM_MIN, miniters=500)
        f, chunk = pos, 0
        while f <= stop:
            if not cap.grab():
                break
            is_want, is_next = f in want, pending is not None and f == pending[0] + 1
            if is_want or is_next:
                ok, frame = cap.retrieve()
                if ok:
                    if is_next:
                        handle(pending[0], pending[1], frame); pending = None
                    if is_want:
                        if motion:
                            pending = (f, frame)
                        else:
                            handle(f, frame, None)
            f += 1; chunk += 1
            if chunk == 500:
                bar.update(chunk); chunk = 0
        bar.update(chunk); bar.close()
        if pending is not None:
            handle(pending[0], pending[1], None)
    cap.release()
    df = pd.DataFrame(rows, columns=["frame", "text", "score", "clock", "motion"])
    df = pd.DataFrame({"frame": frames}).merge(df, on="frame", how="left")
    df["text"] = df["text"].fillna(""); df["score"] = df["score"].fillna(0.0)
    return df, crops


def apply_confidence(df):
    """(Re)derive `clock` from the stored text + score with the current MIN_OCR_SCORE, so a cached scan
    follows a changed threshold without rescanning. Adds `low_conf` = parsed as a clock but rejected."""
    df = df.copy()
    parsed = df["text"].map(parse_clock)
    ok = parsed.notna() & (df["score"] >= MIN_OCR_SCORE)
    df["clock"] = np.where(ok, parsed.astype(float), np.nan)
    df["low_conf"] = parsed.notna() & ~ok
    return df


def cached_scan(source, frames, roi, motion, mode, tag, position=0):
    """scan_video with a parquet cache keyed by video, roi, frames and OCR backend."""
    key = f"{source}_{tag}_{mode}_{int(frames[0])}_{int(frames[-1])}_{len(frames)}_{'x'.join(map(str, roi))}_{OCR_BACKEND}"
    path = CACHE_DIR / f"samples_{key}.parquet"
    if REUSE_CACHE and path.exists():
        print(f"[{source}/{tag}] cached: {path.name}")
        return apply_confidence(pd.read_parquet(path)), []
    df, crops = scan_video(VIDEO[source], frames, roi, motion=motion, mode=mode,
                           desc=f"{source} {tag}", position=position, keep_crops=40)
    try:
        df.to_parquet(path, index=False)
    except Exception as exc:  # pyarrow missing: cache is optional
        print("cache not written:", exc)
    df = apply_confidence(df)
    n_low = int(df["low_conf"].sum())
    print(f"[{source}/{tag}] {np.isfinite(df['clock']).sum()} readings at confidence >= {MIN_OCR_SCORE}, "
          f"{n_low} clock readings rejected below it")
    return df, crops


def scan_both(jobs):
    """jobs: {source: (frames, roi, motion, mode, tag)} -> {source: (df, crops)}; parallel if allowed."""
    if PARALLEL and len(jobs) > 1:
        with ThreadPoolExecutor(max_workers=len(jobs)) as ex:
            futs = {s: ex.submit(cached_scan, s, *a, position=i) for i, (s, a) in enumerate(jobs.items())}
            return {s: f.result() for s, f in futs.items()}
    return {s: cached_scan(s, *a) for s, a in jobs.items()}

## 5. Offset fitting: clock readings → segments → halves → kickoffs

* `vote_offset`: each reading gives `offset ∈ (t−c−1, t−c]`; the point covered by the most intervals
  is the offset, and the width of the winning region is its precision.
* `mark_frozen`: a value seen for more than ~1 s means the clock is held (pre-match, halftime, 45:00 hold).
* `fit_timeline`: rolling-median outlier rejection → split on offset jumps → drop tiny runs → merge
  neighbours with the same offset (replays in between don't change it) → vote → assign halves.

In [ ]:
def mark_frozen(df):
    c, t = df["clock"].to_numpy(), df["t"].to_numpy()
    idx = np.flatnonzero(np.isfinite(c))
    frozen = np.zeros(len(df), dtype=bool)
    start = 0
    for k in range(1, len(idx) + 1):
        if k == len(idx) or c[idx[k]] != c[idx[start]]:
            grp = idx[start:k]
            if t[grp[-1]] - t[grp[0]] > FROZEN_SPAN_SEC:
                frozen[grp] = True
            start = k
    return frozen


def vote_offset(raw):
    """(offset, precision_sec, n_votes) maximising the number of intervals (raw-1, raw] that contain it."""
    r = np.sort(np.asarray(raw, dtype=np.float64))
    lo = np.searchsorted(r, r, side="left")
    hi = np.searchsorted(r, r + 1.0, side="left")
    cov = hi - lo
    j = int(np.argmax(cov))
    members = r[lo[j]:hi[j]]
    a, b = members.max() - 1.0, members.min()
    return (a + b) / 2.0, b - a, int(cov[j])


def fit_timeline(df, fps, min_run=None):
    """Segments [{half, t_first, t_last, offset, ...}] from clock samples (columns t, clock, frozen)."""
    min_run = min_run or MIN_RUN_SAMPLES
    eps = 1.0 / fps + 1e-3
    d = df[np.isfinite(df["clock"]) & ~df["frozen"]].sort_values("t")
    if len(d) < min_run:
        return []
    t, c = d["t"].to_numpy(), d["clock"].to_numpy()
    raw = t - c
    local = pd.Series(raw).rolling(9, center=True, min_periods=3).median().to_numpy()
    keep = np.abs(raw - local) <= 1.0 + eps
    t, c, raw, local = t[keep], c[keep], raw[keep], local[keep]
    cuts = np.flatnonzero(np.abs(np.diff(local)) > OFFSET_TOL_SEC) + 1
    runs = [r for r in np.split(np.arange(len(raw)), cuts) if len(r) >= min_run]
    merged = []
    for r in runs:
        if merged and abs(np.median(raw[merged[-1]]) - np.median(raw[r])) <= OFFSET_TOL_SEC:
            merged[-1] = np.concatenate([merged[-1], r])
        else:
            merged.append(r)
    segs = []
    for r in merged:
        off, width, nv = vote_offset(raw[r])
        inl = (raw[r] - 1.0 - eps < off) & (off <= raw[r] + eps)
        segs.append({"t_first": float(t[r][0]), "t_last": float(t[r][-1]), "offset": float(off),
                     "precision_sec": float(width), "n_reads": int(len(r)), "n_inliers": int(inl.sum()),
                     "inlier_ratio": float(inl.mean()), "clock_first": float(c[r][0]), "clock_last": float(c[r][-1])})
    # halves: a jump in offset whose new segment starts near the next half's start clock
    starts = sorted(HALF_START_CLOCK.items())
    half = max([h for h, s in starts if s - HALF_START_TOL_SEC <= segs[0]["clock_first"]] or [1])
    segs[0]["half"] = half
    for prev, s in zip(segs, segs[1:]):
        nxt = half + 1
        if (nxt in HALF_START_CLOCK and abs(s["clock_first"] - HALF_START_CLOCK[nxt]) <= HALF_START_TOL_SEC
                and s["offset"] - prev["offset"] >= MIN_HALFTIME_JUMP_SEC):
            half = nxt
        s["half"] = half
    return segs


class Timeline:
    """video time <-> (half, match clock) for one source, from its segments."""

    def __init__(self, segs, fps, n_frames, source):
        if not segs:
            raise RuntimeError(f"{source}: no clock segment found. Check the ROI preview and the OCR read rate")
        self.segs, self.fps, self.n_frames, self.source = segs, fps, n_frames, source
        self.off = np.array([s["offset"] for s in segs])
        self.half = np.array([s["half"] for s in segs])
        self.bounds = np.array([(a["t_last"] + b["t_first"]) / 2 for a, b in zip(segs, segs[1:])])

    def seg_index(self, t):
        return np.searchsorted(self.bounds, np.asarray(t, dtype=np.float64), side="right")

    def to_match(self, t):
        """(half, clock_sec) arrays for video times t; clock may fall outside the match (pre/post/halftime)."""
        k = self.seg_index(t)
        return self.half[k], np.asarray(t, dtype=np.float64) - self.off[k]

    def from_match(self, half, clock, extra_offset=None):
        """Video time showing (half, clock); NaN where no segment of that half covers it."""
        half = np.atleast_1d(np.asarray(half)); clock = np.atleast_1d(np.asarray(clock, dtype=np.float64))
        out = np.full(clock.shape, np.nan)
        for k in range(len(self.segs)):
            sel = half == self.half[k]
            if not sel.any():
                continue
            off = self.off[k] + (0.0 if extra_offset is None else extra_offset.get(int(self.half[k]), 0.0))
            tt = clock[sel] + off
            ok = self.seg_index(tt) == k
            tmp = out[sel]; tmp[ok] = tt[ok]; out[sel] = tmp
        return out

    def kickoffs(self):
        ko = {}
        for h, start in HALF_START_CLOCK.items():
            ks = np.flatnonzero(self.half == h)
            if len(ks):
                t = self.off[ks[0]] + start
                # first frame whose match time is >= the start clock (offset is mid-interval, so ceil)
                ko[h] = {"time_sec": float(t), "frame": int(math.ceil(t * self.fps - 1e-6))}
        return ko

## 6. Provided data helpers: broadcast homography `valid` and ATD players

Both are automatic outputs supplied by the organisers (DATA.md), so they're fair inputs. The ATD
parser reads only the frames it's asked for. A raw line is `<frame_id>:<x,y,s;...>:<ball>`, normalised
0–1, with `frame_id = video_frame + 1`.

In [ ]:
def load_hom_valid(source, n_frames):
    path = asset(source, "homography")
    if not USE_HOMOGRAPHY or not path.exists():
        return None
    tab = pd.read_csv(path, usecols=["frame", "valid"])
    v = np.zeros(n_frames, dtype=bool)
    fr = tab["frame"].to_numpy()
    ok = (fr >= 0) & (fr < n_frames)
    v[fr[ok]] = tab["valid"].to_numpy()[ok].astype(bool)
    print(f"[{source}] homography valid on {v.mean():.1%} of frames")
    return v


def load_atd_players(source, wanted, n_frames):
    """{video_frame: (k, 2) normalised player xy} for the wanted frames. Cached per source and frame set."""
    path = asset(source, "atd_raw")
    if not USE_ATD or not path.exists():
        return None
    wanted = np.unique(np.asarray(wanted, dtype=np.int64))
    wanted = wanted[(wanted >= 0) & (wanted < n_frames)]
    key = f"atd_{source}_{len(wanted)}_{int(wanted[0]) if len(wanted) else 0}_{int(wanted[-1]) if len(wanted) else 0}.npz"
    cpath = CACHE_DIR / key
    if REUSE_CACHE and cpath.exists():
        z = np.load(cpath)
        fr, ptr, xy = z["frames"], z["ptr"], z["xy"]
        return {int(f): xy[ptr[i]:ptr[i + 1]] for i, f in enumerate(fr)}
    want = set(wanted.tolist())
    out = {}
    with open(path, "rb") as fh:
        for line in tqdm(fh, total=n_frames, desc=f"ATD {source}", unit="ln", mininterval=max(2.0, TQDM_MIN), miniters=5000):
            i = line.find(b":")
            if i <= 0:
                continue
            try:
                f = int(line[:i]) - 1
            except ValueError:
                continue
            if f not in want:
                continue
            parts = line.decode("ascii", "replace").strip().split(":")
            if len(parts) != 3:
                continue
            pts = []
            for g in parts[1].split(";"):
                if not g or g[0] == "N":
                    continue
                v = g.split(",")
                try:
                    x, y = float(v[0]), float(v[1])
                except (ValueError, IndexError):
                    continue
                if math.isfinite(x) and math.isfinite(y):
                    pts.append((x, y))
            out[f] = np.asarray(pts, dtype=np.float32).reshape(-1, 2)
    fr = np.array(sorted(out), dtype=np.int64)
    ptr = np.concatenate([[0], np.cumsum([len(out[f]) for f in fr])]).astype(np.int64)
    xy = np.concatenate([out[f] for f in fr]) if len(fr) else np.zeros((0, 2), np.float32)
    np.savez_compressed(cpath, frames=fr, ptr=ptr, xy=xy)
    return out


_VARIANTS = {"identity": (False, False), "flip_x": (True, False), "flip_y": (False, True), "rot180": (True, True)}

def _orient(p, variant):
    fx, fy = _VARIANTS[variant]
    q = p.copy()
    if fx: q[:, 0] = 1.0 - q[:, 0]
    if fy: q[:, 1] = 1.0 - q[:, 1]
    return q


def chamfer_m(pb, pt, variant="identity"):
    """Median distance (m) from each broadcast-tracked player to the nearest tactical-tracked player."""
    if pb is None or pt is None or len(pb) < ATD_MIN_PLAYERS or len(pt) < ATD_MIN_PLAYERS:
        return np.nan
    a = _orient(pb, variant) * PITCH_M
    b = pt * PITCH_M
    d = np.sqrt(((a[:, None, :] - b[None, :, :]) ** 2).sum(-1)).min(1)
    return float(np.median(d))

## 7. Broadcast shot labels

| label | rule |
|---|---|
| `OFF_MATCH` | estimated match clock outside [kickoff, end of half]: pre-match, halftime, after the final whistle |
| `LIVE_WIDE` | clock visible and consistent, homography valid; or no clock but the ATD agrees with tactical |
| `LIVE_CLOSE` | clock visible, no homography (zoom); or a short clock-less stretch with no replay evidence |
| `REPLAY` | clock-less stretch with ATD "different moment" votes or slow motion |
| `UNKNOWN` | none of the above; excluded from the mapping |

Only `LIVE_*` frames get a `tactical_frame`. The clock keeps running during a replay, so mapping one by
offset would point to the wrong tactical frame.

In [ ]:
LABELS = ["OFF_MATCH", "LIVE_WIDE", "LIVE_CLOSE", "REPLAY", "UNKNOWN"]
LABEL_CODE = {k: i for i, k in enumerate(LABELS)}


def half_end_clocks(tl_T, df_T):
    """End-of-half match clock: the tactical final-whistle frames from games.py if present, else the last reading."""
    ends = {}
    if RUN_MODE != "selftest":
        for h, _first, last in HALF_VIDEO_BLOCKS:
            hh, clk = tl_T.to_match(np.array([last / tl_T.fps]))
            if int(hh[0]) == h:
                ends[h] = float(clk[0])
    for h in HALF_START_CLOCK:
        if h not in ends:
            hh, _ = tl_T.to_match(df_T["t"].to_numpy())
            sel = (hh == h) & np.isfinite(df_T["clock"].to_numpy()) & ~df_T["frozen"].to_numpy()
            if sel.any():
                ends[h] = float(df_T["clock"].to_numpy()[sel].max() + 1.0)
    return ends


def map_to_tactical(half, clock, tl_T, pair_extra):
    """Tactical frame (float, NaN if outside) for (half, clock), with the per-half pair correction."""
    tt = tl_T.from_match(half, clock, extra_offset=pair_extra)
    return tt * tl_T.fps


def classify_samples(df_B, tl_B, tl_T, end_clock, pair_extra, hom_valid, atd_B, atd_T, variant):
    """Adds half, match_clock, in_match, clock_ok, atd_m, slowmo, label, confidence, tactical_frame to df_B."""
    df = df_B.copy()
    half, clk = tl_B.to_match(df["t"].to_numpy())
    df["half"], df["match_clock"] = half, clk
    start = np.array([HALF_START_CLOCK.get(int(h), np.nan) for h in half])
    end = np.array([end_clock.get(int(h), np.inf) for h in half])
    df["in_match"] = (clk >= start) & (clk <= end)
    obs = df["clock"].to_numpy()
    df["clock_ok"] = np.isfinite(obs) & (np.abs(obs - np.floor(clk)) <= 1.0) & ~df["frozen"].to_numpy()
    tf = map_to_tactical(half, clk, tl_T, pair_extra)
    df["tactical_frame"] = tf
    hv = np.ones(len(df), bool) if hom_valid is None else hom_valid[np.clip(df["frame"].to_numpy(), 0, len(hom_valid) - 1)]
    df["hom_valid"] = hv
    # ATD cross-check at the mapped tactical frame
    atd = np.full(len(df), np.nan)
    if atd_B is not None and atd_T is not None:
        for i, (fb, ft) in enumerate(zip(df["frame"].to_numpy(), tf)):
            if np.isfinite(ft):
                atd[i] = chamfer_m(atd_B.get(int(fb)), atd_T.get(int(round(ft))), variant.get(int(half[i]), "identity"))
    df["atd_m"] = atd
    mot = df["motion"].to_numpy()
    live_mot = mot[df["clock_ok"].to_numpy() & df["in_match"].to_numpy() & np.isfinite(mot)]
    base = float(np.median(live_mot)) if len(live_mot) else np.nan
    df["slowmo"] = np.isfinite(mot) & np.isfinite(base) & (mot < SLOWMO_REL * base)

    lab = np.array(["UNKNOWN"] * len(df), dtype=object)
    conf = np.zeros(len(df))
    ok = df["clock_ok"].to_numpy()
    lab[ok & hv], conf[ok & hv] = "LIVE_WIDE", 1.0
    lab[ok & ~hv], conf[ok & ~hv] = "LIVE_CLOSE", 0.9
    atd_live = np.isfinite(atd) & (atd < ATD_LIVE_M)
    atd_rep = np.isfinite(atd) & (atd > ATD_REPLAY_M)
    # clock-less stretches, decided as a whole
    step_sec = (np.median(np.diff(df["t"].to_numpy())) if len(df) > 1 else 0.5)
    idx = np.flatnonzero(~ok)
    if len(idx):
        breaks = np.flatnonzero(np.diff(idx) > 1) + 1
        for run in np.split(idx, breaks):
            dur = df["t"].to_numpy()[run[-1]] - df["t"].to_numpy()[run[0]] + step_sec
            has_atd = np.isfinite(atd[run])
            rep_share = atd_rep[run][has_atd].mean() if has_atd.any() else 0.0
            live_share = atd_live[run][has_atd].mean() if has_atd.any() else 0.0
            slow_share = df["slowmo"].to_numpy()[run].mean()
            # order matters: ATD evidence first; a short cut is a live close-up (replays are longer);
            # only then is low motion trusted as slow motion (a static close-up also has low motion)
            if rep_share >= ATD_REPLAY_SHARE:
                default, dconf = "REPLAY", 0.7
            elif live_share >= 0.5:
                default, dconf = "LIVE", 0.6
            elif dur <= SHORT_GAP_SEC:
                default, dconf = "LIVE_CLOSE", 0.5
            elif slow_share >= SLOWMO_SHARE:
                default, dconf = "REPLAY", 0.6
            else:
                default, dconf = "UNKNOWN", 0.0
            for i in run:
                if atd_live[i] and default != "REPLAY":
                    lab[i], conf[i] = ("LIVE_WIDE" if hv[i] else "LIVE_CLOSE"), 0.8
                elif atd_rep[i]:
                    lab[i], conf[i] = "REPLAY", 0.8
                elif default == "LIVE":
                    lab[i], conf[i] = ("LIVE_WIDE" if hv[i] else "LIVE_CLOSE"), dconf
                else:
                    lab[i], conf[i] = default, dconf
    # safeguard: clock visible but the ATD consistently says "different moment" -> replay with the clock kept on
    idx = np.flatnonzero(ok)
    if len(idx):
        for run in np.split(idx, np.flatnonzero(np.diff(idx) > 1) + 1):
            has_atd = np.isfinite(atd[run])
            if has_atd.sum() >= 3 and atd_rep[run][has_atd].mean() >= 0.6:
                lab[run], conf[run] = "REPLAY", 0.7
    off = ~df["in_match"].to_numpy()
    lab[off], conf[off] = "OFF_MATCH", 1.0
    df["label"], df["confidence"] = lab, conf
    return df


def choose_variant(df, atd_B, atd_T):
    """Per half, the pitch orientation under which broadcast and tactical ATD agree best on live frames."""
    out = {}
    for h in HALF_START_CLOCK:
        sel = df[(df["half"] == h) & df["clock_ok"] & df["in_match"] & df["hom_valid"] & np.isfinite(df["tactical_frame"])]
        if sel.empty:
            continue
        sel = sel.sample(min(len(sel), 200), random_state=SEED)
        score = {v: np.nanmedian([chamfer_m(atd_B.get(int(fb)), atd_T.get(int(round(ft))), v)
                                  for fb, ft in zip(sel["frame"], sel["tactical_frame"])] or [np.nan])
                 for v in _VARIANTS}
        score = {k: v for k, v in score.items() if np.isfinite(v)}
        if score:
            out[h] = min(score, key=score.get)
    return out


def atd_refine(df, atd_B, atd_T, variant, fps_T):
    """Per half, the tactical shift (frames) minimising the ATD distance on live wide frames."""
    report, extra = {}, {}
    shifts = np.arange(-ATD_REFINE_MAX_SHIFT, ATD_REFINE_MAX_SHIFT + 1)
    for h in HALF_START_CLOCK:
        sel = df[(df["half"] == h) & df["clock_ok"] & df["in_match"] & df["hom_valid"] & np.isfinite(df["tactical_frame"])]
        sel = sel[[atd_B.get(int(f)) is not None and len(atd_B[int(f)]) >= ATD_MIN_PLAYERS for f in sel["frame"]]]
        if len(sel) < 20:
            report[h] = {"status": "skipped", "n": int(len(sel))}
            continue
        sel = sel.sample(min(len(sel), ATD_REFINE_SAMPLES), random_state=SEED)
        fb, ft = sel["frame"].to_numpy(), np.round(sel["tactical_frame"].to_numpy()).astype(int)
        v = variant.get(h, "identity")
        costs = np.array([np.nanmedian([chamfer_m(atd_B.get(int(b)), atd_T.get(int(t + s)), v) for b, t in zip(fb, ft)])
                          for s in tqdm(shifts, desc=f"ATD refine half {h}", leave=False)])
        j = int(np.nanargmin(costs))
        accepted = bool(costs[j] <= ATD_LIVE_M and costs[j] < 0.85 * np.nanmedian(costs))
        report[h] = {"status": "accepted" if accepted else "rejected", "best_shift_frames": int(shifts[j]),
                     "cost_at_best_m": float(costs[j]), "cost_at_zero_m": float(costs[shifts == 0][0]),
                     "median_cost_m": float(np.nanmedian(costs)), "n": int(len(sel)), "variant": v}
        if accepted:
            extra[h] = float(shifts[j]) / fps_T
    return extra, report

## 8. Synthetic videos (selftest only)

In [ ]:
def _draw_clock(img, roi, text, style):
    x, y, w, h = roi
    bg, fg = ((255, 255, 255), (20, 20, 30)) if style == "broadcast" else ((60, 60, 60), (255, 255, 255))
    cv2.rectangle(img, (x, y), (x + w, y + h), bg, -1)
    scale = 0.3
    while True:
        (tw, th), _ = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, scale + 0.05, 2)
        if tw > w * 0.85 or th > h * 0.65:
            break
        scale += 0.05
    (tw, th), _ = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, scale, 2)
    cv2.putText(img, text, (x + (w - tw) // 2, y + (h + th) // 2), cv2.FONT_HERSHEY_SIMPLEX, scale, fg, 2, cv2.LINE_AA)


def _scene(tau, size, zoom=1.0, seed=7):
    W, H = size
    img = np.zeros((H, W, 3), np.uint8); img[:] = (40, 120, 40)
    for k in range(0, W, 80):
        cv2.rectangle(img, (k, 0), (k + 40, H), (45, 135, 45), -1)
    rng = np.random.default_rng(seed)
    p0 = rng.uniform([0, 0], [W, H], size=(14, 2)); v = rng.uniform(-90, 90, size=(14, 2))
    p = (p0 + v * tau) % [W, H]
    for i, (px, py) in enumerate(p):
        cx, cy = int((px - W / 2) * zoom + W / 2), int((py - H / 2) * zoom + H / 2)
        cv2.circle(img, (cx, cy), int(9 * zoom), (30, 30, 200) if i % 2 else (230, 230, 230), -1)
    return img


SYN_TRUE_ROI_REF = dict(CLOCK_ROI_REF)     # synthetic clocks are drawn exactly where the fixed boxes look

def make_synthetic(path, source, duration, plan, fps=25, size=(1280, 720)):
    """plan: list of (kind, t0, t1, arg). kinds: none | frozen(text) | live(offset) | replay(src_t0) | close(offset)."""
    info = {"width": size[0], "height": size[1]}
    x, y, w, h = SYN_TRUE_ROI_REF[source]; k = size[0] / REF_SIZE[0]
    roi = tuple(int(round(v * k)) for v in (x, y, w, h))
    wr = cv2.VideoWriter(str(path), cv2.VideoWriter_fourcc(*"mp4v"), fps, size)
    n = int(round(duration * fps))
    for f in range(n):
        t = f / fps
        kind, t0, t1, arg = next((p for p in plan if p[1] <= t < p[2]), ("none", 0, 0, None))
        if kind == "replay":
            tau = arg + np.floor((t - t0) * fps / 3) / fps          # 1/3 speed, every frame shown 3x
            img = _scene(tau, size)
        elif kind == "close":
            img = _scene(t, size, zoom=2.5)
        else:
            img = _scene(t, size)
        if kind == "live":
            m = max(0, int(math.floor(t - arg + 1e-9)))
            _draw_clock(img, roi, f"{m // 60:02d}:{m % 60:02d}", source)
        elif kind == "frozen":
            _draw_clock(img, roi, arg, source)
        wr.write(img)
    wr.release()


SYN_CLOCK_LAG_SEC = 0.4      # the synthetic broadcast clock runs 10 frames behind the true instant
SYN_REPLAY_LAG_SEC = 12.0


def _players(m, n=20, seed=11):
    """Normalised positions of n synthetic players at match time m (bouncing, ~4 m/s)."""
    rng = np.random.default_rng(seed)
    p0 = rng.uniform(0.05, 0.95, size=(n, 2)); v = rng.uniform(-0.05, 0.05, size=(n, 2))
    u = p0 + v * m
    return 1.0 - np.abs(np.mod(u, 2.0) - 1.0)


def make_synthetic_data(source, duration, plan, fps=25):
    """Synthetic ATD (+ broadcast homography `valid`) consistent with the synthetic videos.
    Broadcast: live frames show the instant clock + SYN_CLOCK_LAG_SEC, 12 of 20 players, and the
    2nd half is in the rotated (rot180) pitch orientation; replays show an instant 12 s earlier."""
    lines, valid = [], []
    rng = np.random.default_rng(3)
    for f in range(int(round(duration * fps))):
        t = f / fps
        kind, t0, t1, arg = next((p for p in plan if p[1] <= t < p[2]), ("none", 0, 0, None))
        pts = None
        if source == "tactical" and kind == "live":
            pts = _players(t - arg)
        elif source == "broadcast" and kind in ("live", "replay"):
            m = (t - arg + SYN_CLOCK_LAG_SEC) if kind == "live" else (t - 3.12 - SYN_REPLAY_LAG_SEC)
            pts = _players(m)[:12]
            if kind == "live" and arg == 11.3:
                pts = 1.0 - pts
        if pts is not None:
            pts = pts + rng.normal(0, 0.003, pts.shape)
        valid.append(int(source == "broadcast" and kind in ("live", "replay")))
        ptxt = ";".join(f"{x:.5f},{y:.5f},0.0" for x, y in pts) if pts is not None else "NaN,NaN,NaN"
        lines.append(f"{f + 1}:{ptxt}:NaN,NaN,NaN\n")
    with open(asset(source, "atd_raw"), "w") as fh:
        fh.writelines(lines)
    if source == "broadcast":
        cols = ["h{}{}".format(r, c) for r in range(3) for c in range(3)]
        hom = pd.DataFrame({"frame": np.arange(len(valid))})
        for c in cols:
            hom[c] = np.nan
        hom["reprojection_error"] = np.nan; hom["valid"] = valid
        hom.to_csv(asset(source, "homography"), index=False)


if RUN_MODE == "selftest":
    SYN = {
        "tactical": dict(duration=100.0, plan=[("frozen", 4.0, 8.36, "00:00"), ("live", 8.36, 54.36, 8.36),
                                               ("frozen", 54.36, 62.2, "00:40"), ("live", 62.2, 92.2, 22.2)]),
        "broadcast": dict(duration=85.0, plan=[("live", 3.12, 20.0, 3.12), ("replay", 20.0, 26.0, 10.0),
                                               ("live", 26.0, 30.0, 3.12), ("close", 30.0, 33.0, 3.12),
                                               ("live", 33.0, 49.12, 3.12), ("live", 51.3, 81.3, 11.3)]),
    }
    for s in SOURCES:
        make_synthetic(VIDEO[s], s, **SYN[s])
        make_synthetic_data(s, **SYN[s])
        print(s, video_info(VIDEO[s]))

## 9. Clock box: pixel grid, check and preview

1. **Pixel grid.** The top-left of 3 frames per video, with real pixel coordinates on the axes and the
   current box in red. If the red box doesn't sit on the clock, read the clock's corners off the axes
   and set `CLOCK_ROI_REF[source] = (x, y, w, h)` in the config (or `SYNC_ROI_BROADCAST="x,y,w,h"`).
   Leave ~5 px of margin and don't include the team name.
2. **Check.** Each box is OCR'd on 24 frames across the video. In smoke/full the notebook **stops here**
   if a box reads a clock on fewer than 30% of them, so a wrong box never costs a long scan. (Some
   probes fall before kickoff, at halftime or in replays, so 60–90% is normal for a correct box.)
3. **Preview.** 8 crops per video with what the OCR read.

In [ ]:
INFO = {s: video_info(VIDEO[s]) for s in SOURCES}
ROI = {s: roi_px(s, INFO[s]) for s in SOURCES}
STEP = {s: SAMPLE_STEP or auto_step(INFO[s]["fps"]) for s in SOURCES}
for s in SOURCES:
    print(f"{s:9s} {INFO[s]}  box={ROI[s]}  step={STEP[s]}  expected frames={EXPECTED_FRAMES[s]}")


def _read_frame(source, f):
    cap = cv2.VideoCapture(str(VIDEO[source])); cap.set(cv2.CAP_PROP_POS_FRAMES, int(f))
    ok, fr = cap.read(); cap.release()
    return fr if ok else None


def show_roi_grid():
    """Top-left of 3 frames per video on real pixel axes (ticks every 50 px, grid every 10 px), box in red."""
    from matplotlib.patches import Rectangle
    from matplotlib.ticker import MultipleLocator
    fig, axes = plt.subplots(len(SOURCES), 3, figsize=(18, 4.2 * len(SOURCES)))
    for r, s in enumerate(SOURCES):
        sx, sy = INFO[s]["width"] / REF_SIZE[0], INFO[s]["height"] / REF_SIZE[1]
        X, Y, W, H = (int(round(v * k)) for v, k in zip(ROI_GRID_REF, (sx, sy, sx, sy)))
        n = INFO[s]["n_frames"]
        for c, frac in enumerate((0.25, 0.5, 0.75)):
            f = int(frac * n); ax = axes[r, c]
            fr = _read_frame(s, f)
            if fr is not None:
                ax.imshow(cv2.cvtColor(fr[Y:Y + H, X:X + W], cv2.COLOR_BGR2RGB), extent=(X, X + W, Y + H, Y))
            x, y, w, h = ROI[s]
            ax.add_patch(Rectangle((x, y), w, h, fill=False, ec="red", lw=1.5))
            ax.xaxis.set_major_locator(MultipleLocator(50)); ax.yaxis.set_major_locator(MultipleLocator(50))
            ax.xaxis.set_minor_locator(MultipleLocator(10)); ax.yaxis.set_minor_locator(MultipleLocator(10))
            ax.grid(which="major", color="yellow", lw=0.6, alpha=0.8); ax.grid(which="minor", color="yellow", lw=0.3, alpha=0.35)
            ax.tick_params(labelsize=7, labeltop=True)
            ax.set_title(f"{s} frame {f}  (red = box {ROI[s]})", fontsize=9)
    plt.tight_layout(); plt.savefig(OUT_DIR / f"sync_{RUN_MODE}_roi_grid.png", dpi=110); plt.show()


ROI_SOURCE = {s: "fixed" for s in SOURCES}
ROI_RATE, PROBE = {}, {}
for s in SOURCES:
    n = INFO[s]["n_frames"]
    PROBE[s] = np.linspace(0.05 * n, 0.95 * n, ROI_CHECK_PROBES).astype(int)
    dfp, _ = scan_video(VIDEO[s], PROBE[s], ROI[s], mode="seek", desc=f"box check {s}")
    ROI_RATE[s] = float(np.isfinite(dfp["clock"]).mean())
    print(f"[{s}] box {ROI[s]} reads a clock on {ROI_RATE[s]:.0%} of {len(PROBE[s])} probe frames")
    if ROI_MODE == "auto" and ROI_RATE[s] < ROI_MIN_RATE:
        roi, hits = locate_clock_roi(s, INFO[s], PROBE[s])
        if roi is not None:
            dfa, _ = scan_video(VIDEO[s], PROBE[s], roi, mode="seek", desc=f"box check {s} (auto)")
            rate_a = float(np.isfinite(dfa["clock"]).mean())
            print(f"[{s}] auto box {roi} ({hits} detections) reads {rate_a:.0%}")
            if rate_a > ROI_RATE[s]:
                ROI[s], ROI_SOURCE[s], ROI_RATE[s] = roi, "auto", rate_a

if SHOW_ROI_GRID:
    show_roi_grid()

fig, axes = plt.subplots(2, 8, figsize=(16, 2.6))
for r, s in enumerate(SOURCES):
    probe = PROBE[s][np.linspace(0, len(PROBE[s]) - 1, 8).astype(int)]
    dfp, _ = scan_video(VIDEO[s], probe, ROI[s], mode="seek", desc=f"preview {s}")
    x, y, w, h = ROI[s]
    for c, f in enumerate(probe):
        fr = _read_frame(s, f); ax = axes[r, c]; ax.axis("off")
        if fr is not None:
            ax.imshow(cv2.cvtColor(fr[y:y + h, x:x + w], cv2.COLOR_BGR2RGB))
        row = dfp[dfp["frame"] == f].iloc[0]
        ax.set_title(f"{s[:5]} {f}\n'{row['text']}' {row['score']:.2f}", fontsize=7)
plt.tight_layout(); plt.savefig(OUT_DIR / f"sync_{RUN_MODE}_roi_preview.png", dpi=110); plt.show()

bad = [s for s in SOURCES if ROI_RATE[s] < ROI_MIN_RATE]
if bad:
    raise RuntimeError(
        "clock box reads too few clocks: " + ", ".join(f"{s} {ROI_RATE[s]:.0%}" for s in bad) + ". Look at the pixel grid "
        f"above, set CLOCK_ROI_REF[...] = (x, y, w, h) around the clock (or env SYNC_ROI_BROADCAST='x,y,w,h') and re-run. "
        f"Stopping here so no long scan runs with a wrong box.")

## 10. Scan the clocks

* `selftest` / `full`: sequential pass over each whole video, a read every `STEP` frames.
* `smoke`: coarse seek every ~20 s over the whole match (segments and halves), then a dense pass over
  the first `SMOKE_DENSE_SEC` after the 1st-half kickoff in both videos (frame-accurate offsets and
  per-frame output).

In [ ]:
def add_time(df, s):
    df = df.sort_values("frame").drop_duplicates("frame").reset_index(drop=True)
    df["t"] = df["frame"] / INFO[s]["fps"]
    df["frozen"] = mark_frozen(df)
    return df

T0 = time.time()
CROPS = {s: [] for s in SOURCES}
DENSE_RANGE = {s: (0, INFO[s]["n_frames"] - 1) for s in SOURCES}

if RUN_MODE in ("selftest", "full"):
    jobs = {s: (np.arange(0, INFO[s]["n_frames"] - 1, STEP[s]), ROI[s], s == "broadcast", "seq", "dense")
            for s in SOURCES}
    res = scan_both(jobs)
    SAMPLES = {s: add_time(res[s][0], s) for s in SOURCES}
    CROPS = {s: res[s][1] for s in SOURCES}
else:  # smoke
    jobs = {}
    for s in SOURCES:
        step = int(round(SMOKE_COARSE_STEP_SEC * INFO[s]["fps"])) + 1     # +1: phases sweep the frames
        jobs[s] = (np.arange(0, INFO[s]["n_frames"] - 1, step), ROI[s], s == "broadcast", "seek", "coarse")
    res = scan_both(jobs)
    coarse = {s: add_time(res[s][0], s) for s in SOURCES}
    tl0 = {s: Timeline(fit_timeline(coarse[s], INFO[s]["fps"], min_run=4), INFO[s]["fps"], INFO[s]["n_frames"], s)
           for s in SOURCES}
    ko_T = tl0["tactical"].kickoffs()[1]["time_sec"]
    t_lo, t_hi = ko_T - 10, ko_T + SMOKE_DENSE_SEC
    h_lo, c_lo = tl0["tactical"].to_match([t_lo]); h_hi, c_hi = tl0["tactical"].to_match([t_hi])
    tb = tl0["broadcast"].from_match(np.r_[h_lo, h_hi], np.r_[c_lo, c_hi])
    if not np.all(np.isfinite(tb)):
        ko_B = tl0["broadcast"].kickoffs()[1]["time_sec"]; tb = np.array([ko_B - 10, ko_B + SMOKE_DENSE_SEC])
    windows = {"tactical": (t_lo, t_hi), "broadcast": (tb[0] - 5, tb[1] + 5)}
    jobs = {}
    for s in SOURCES:
        a = max(0, int(windows[s][0] * INFO[s]["fps"])); b = min(INFO[s]["n_frames"] - 2, int(windows[s][1] * INFO[s]["fps"]))
        DENSE_RANGE[s] = (a, b)
        jobs[s] = (np.arange(a, b, STEP[s]), ROI[s], s == "broadcast", "seq", "dense")
    res = scan_both(jobs)
    SAMPLES = {s: add_time(pd.concat([coarse[s], res[s][0]]), s) for s in SOURCES}
    CROPS = {s: res[s][1] for s in SOURCES}

for s in SOURCES:
    d = SAMPLES[s]
    low = int(d["low_conf"].sum()) if "low_conf" in d else 0
    print(f"{s:9s} samples={len(d):6d}  clock read (conf >= {MIN_OCR_SCORE})={np.isfinite(d['clock']).mean():.1%}  "
          f"rejected low-confidence={low}  frozen={d['frozen'].mean():.1%}")
print(f"scan done in {time.time() - T0:.0f}s")

## 11. Fit timelines, kickoffs, end of halves

In [ ]:
SEGS = {s: fit_timeline(SAMPLES[s], INFO[s]["fps"]) for s in SOURCES}
TL = {s: Timeline(SEGS[s], INFO[s]["fps"], INFO[s]["n_frames"], s) for s in SOURCES}
KICKOFF = {s: TL[s].kickoffs() for s in SOURCES}
END_CLOCK = half_end_clocks(TL["tactical"], SAMPLES["tactical"])

for s in SOURCES:
    print(f"\n{s}: {len(SEGS[s])} segment(s)")
    print(pd.DataFrame(SEGS[s]).round(3).to_string(index=False))
    for h, k in KICKOFF[s].items():
        print(f"  kickoff half {h}: frame {k['frame']}  (t={k['time_sec']:.3f}s)")
print("\nend-of-half clock:", {h: f"{int(c // 60):02d}:{c % 60:05.2f}" for h, c in END_CLOCK.items()})

## 12. Label broadcast samples, ATD cross-check and pair refinement

In [ ]:
n_B = INFO["broadcast"]["n_frames"]
HOM_VALID_B = load_hom_valid("broadcast", n_B)
PAIR_EXTRA, REFINE_REPORT, VARIANT = {}, {}, {}
ATD_B = ATD_T = None

base = classify_samples(add_time(SAMPLES["broadcast"], "broadcast"), TL["broadcast"], TL["tactical"], END_CLOCK,
                        {}, HOM_VALID_B, None, None, {})
if USE_ATD and asset("broadcast", "atd_raw").exists() and asset("tactical", "atd_raw").exists():
    ATD_B = load_atd_players("broadcast", base["frame"].to_numpy(), n_B)
    tf = base["tactical_frame"].to_numpy()
    tf = np.round(tf[np.isfinite(tf)]).astype(np.int64)
    wanted_T = np.unique((tf[:, None] + np.arange(-ATD_REFINE_MAX_SHIFT, ATD_REFINE_MAX_SHIFT + 1)[None, :]).ravel())
    ATD_T = load_atd_players("tactical", wanted_T, INFO["tactical"]["n_frames"])
    VARIANT = choose_variant(base, ATD_B, ATD_T)
    print("ATD orientation per half:", VARIANT)
    PAIR_EXTRA, REFINE_REPORT = atd_refine(base, ATD_B, ATD_T, VARIANT, INFO["tactical"]["fps"])
    print("ATD refinement:", json.dumps(REFINE_REPORT, indent=1))

LAB_B = classify_samples(add_time(SAMPLES["broadcast"], "broadcast"), TL["broadcast"], TL["tactical"], END_CLOCK,
                         PAIR_EXTRA, HOM_VALID_B, ATD_B, ATD_T, VARIANT)
print(LAB_B["label"].value_counts().to_string())

## 13. Per-frame tables

* **broadcast → tactical**: every broadcast frame gets its half, match clock (seconds; the 2nd half
  starts at 2700), seconds since its half's kickoff, label, confidence and `tactical_frame`
  (`-1` unless live). Labels come from the nearest sample, so boundaries are accurate to ±`STEP/2` frames.
* **tactical → broadcast**: every tactical frame gets its half, match clock and the live broadcast
  frame showing the same instant (`-1` if the broadcast is in a replay, a gap or a cut).

In smoke mode the tables cover only the dense window.

In [ ]:
def nearest_sample(sample_frames, frames):
    j = np.clip(np.searchsorted(sample_frames, frames), 1, len(sample_frames) - 1)
    left, right = sample_frames[j - 1], sample_frames[j]
    return np.where(frames - left <= right - frames, j - 1, j)

def broadcast_table():
    a, b = DENSE_RANGE["broadcast"]
    fr = np.arange(a, b + 1)
    t = fr / INFO["broadcast"]["fps"]
    half, clk = TL["broadcast"].to_match(t)
    sf = LAB_B["frame"].to_numpy()
    j = nearest_sample(sf, fr)
    lab = LAB_B["label"].to_numpy()[j].copy(); conf = LAB_B["confidence"].to_numpy()[j].copy()
    start = np.array([HALF_START_CLOCK.get(int(h), np.nan) for h in half])
    end = np.array([END_CLOCK.get(int(h), np.inf) for h in half])
    off = ~((clk >= start) & (clk <= end))
    lab[off], conf[off] = "OFF_MATCH", 1.0
    tfn = map_to_tactical(half, clk, TL["tactical"], PAIR_EXTRA)
    live = np.isin(lab, ["LIVE_WIDE", "LIVE_CLOSE"]) & np.isfinite(tfn)
    tfr = np.where(live, np.round(np.nan_to_num(tfn, nan=-1)), -1).astype(np.int64)
    tfr[(tfr < 0) | (tfr >= INFO["tactical"]["n_frames"])] = -1
    return pd.DataFrame({"broadcast_frame": fr, "half": half.astype(int), "match_clock_sec": clk,
                         "half_elapsed_sec": clk - start, "label": lab, "confidence": conf, "tactical_frame": tfr})

def tactical_table(btab):
    a, b = (DENSE_RANGE["tactical"] if RUN_MODE == "smoke" else (0, INFO["tactical"]["n_frames"] - 1))
    fr = np.arange(a, b + 1)
    half, clk = TL["tactical"].to_match(fr / INFO["tactical"]["fps"])
    start = np.array([HALF_START_CLOCK.get(int(h), np.nan) for h in half])
    end = np.array([END_CLOCK.get(int(h), np.inf) for h in half])
    in_match = (clk >= start) & (clk <= end)
    # inverse map: tactical_time = clock + off_T + extra  =>  the broadcast shows clock - extra
    extra_vec = np.array([PAIR_EXTRA.get(int(h), 0.0) for h in half])
    tb = TL["broadcast"].from_match(half, clk - extra_vec)
    bf = np.round(np.nan_to_num(tb * INFO["broadcast"]["fps"], nan=-1)).astype(np.int64)
    ok = in_match & (bf >= btab["broadcast_frame"].iloc[0]) & (bf <= btab["broadcast_frame"].iloc[-1])
    lab = np.full(len(fr), "", dtype=object)
    lab[ok] = btab["label"].to_numpy()[bf[ok] - btab["broadcast_frame"].iloc[0]]
    live = ok & np.isin(lab, ["LIVE_WIDE", "LIVE_CLOSE"])
    bf = np.where(live, bf, -1)
    return pd.DataFrame({"tactical_frame": fr, "half": half.astype(int), "match_clock_sec": clk,
                         "half_elapsed_sec": clk - start, "in_match": in_match, "broadcast_frame": bf})

BTAB = broadcast_table()
TTAB = tactical_table(BTAB)
print("broadcast frames:", len(BTAB), "|", BTAB["label"].value_counts(normalize=True).round(3).to_dict())
print("tactical frames :", len(TTAB), "| with a live broadcast frame:", f"{(TTAB['broadcast_frame'] >= 0).mean():.1%}")

## 14. Checks (smoke / full / selftest)

In [ ]:
CHECKS = []
def check(name, passed, detail="", critical=True):
    CHECKS.append({"name": name, "passed": bool(passed), "critical": critical, "detail": str(detail)})

fps_T, fps_B = INFO["tactical"]["fps"], INFO["broadcast"]["fps"]
for s in SOURCES:
    check(f"{s}: frame count = docs", INFO[s]["n_frames"] == EXPECTED_FRAMES[s],
          f"{INFO[s]['n_frames']} vs {EXPECTED_FRAMES[s]} (FAQ: a different count means a different decode)", critical=False)
    halves = sorted({sg["half"] for sg in SEGS[s]})
    check(f"{s}: both halves found", halves == sorted(HALF_START_CLOCK), halves)
    for sg in SEGS[s]:
        check(f"{s}: segment half {sg['half']} @{sg['t_first']:.0f}s inlier ratio", sg["inlier_ratio"] >= (0.9 if s == "tactical" else 0.8),
              f"{sg['inlier_ratio']:.3f} of {sg['n_reads']}", critical=False)
        check(f"{s}: segment half {sg['half']} @{sg['t_first']:.0f}s offset precision <= 2 frames",
              sg["precision_sec"] <= 2.0 / INFO[s]["fps"] + 1e-6, f"{sg['precision_sec'] * 1000:.0f} ms", critical=False)

dT = SAMPLES["tactical"]
hh, cc = TL["tactical"].to_match(dT["t"].to_numpy())
inm = (cc >= np.array([HALF_START_CLOCK[int(h)] for h in hh])) & (cc <= np.array([END_CLOCK.get(int(h), np.inf) for h in hh]))
rate_T = np.isfinite(dT["clock"].to_numpy()[inm]).mean() if inm.any() else 0.0
check("tactical: OCR read rate during the match >= 90%", rate_T >= 0.90, f"{rate_T:.1%}")
live_B = LAB_B["in_match"].to_numpy()
rate_B = np.isfinite(LAB_B["clock"].to_numpy()[live_B]).mean() if live_B.any() else 0.0
check("broadcast: OCR read rate during the match >= 50%", rate_B >= 0.50, f"{rate_B:.1%}")

for h, exp in EXPECTED_TACTICAL_KICKOFF.items():
    got = KICKOFF["tactical"].get(h, {}).get("frame")
    check(f"tactical kickoff half {h} vs docs ({exp})", got is not None and abs(got - exp) <= KICKOFF_TOL_FRAMES,
          f"measured {got}, delta {None if got is None else got - exp} frames")

live = BTAB[BTAB["tactical_frame"] >= 0]
mono = all((np.diff(g["tactical_frame"].to_numpy()) >= 0).mean() >= 0.999 for _, g in live.groupby("half")) if len(live) else False
check("broadcast->tactical monotonic within each half", mono)
if len(live):
    smp = live.sample(min(500, len(live)), random_state=SEED)
    back = TTAB.set_index("tactical_frame")["broadcast_frame"].reindex(smp["tactical_frame"]).to_numpy()
    okb = back >= 0
    err = np.abs(back[okb] - smp["broadcast_frame"].to_numpy()[okb])
    check("round trip broadcast->tactical->broadcast within 1 frame", okb.any() and np.percentile(err, 99) <= 1,
          f"p99 {np.percentile(err, 99) if okb.any() else 'n/a'} frames over {okb.sum()} frames")
if ATD_B is not None:
    lw = LAB_B[(LAB_B["label"] == "LIVE_WIDE") & np.isfinite(LAB_B["atd_m"])]
    med = float(lw["atd_m"].median()) if len(lw) else np.nan
    check(f"ATD agreement on LIVE_WIDE (median < {ATD_LIVE_M} m)", np.isfinite(med) and med < ATD_LIVE_M, f"{med:.2f} m", critical=False)

if RUN_MODE == "selftest":
    exp_off = {"tactical": {1: 8.36, 2: 22.2}, "broadcast": {1: 3.12, 2: 11.3}}
    for s in SOURCES:
        for sg in SEGS[s]:
            check(f"selftest {s} half {sg['half']} offset within 1 frame", abs(sg["offset"] - exp_off[s][sg["half"]]) <= 1 / 25 + 1e-6,
                  f"{sg['offset']:.3f} vs {exp_off[s][sg['half']]}")
    for (lo, hi, want, share) in [(20.6, 25.4, "REPLAY", 0.8), (30.6, 32.4, "LIVE_CLOSE", 0.8), (5, 19, "LIVE_WIDE", 0.95),
                                  (0, 2.5, "OFF_MATCH", 0.95), (49.6, 50.8, "OFF_MATCH", 0.95), (55, 80, "LIVE_WIDE", 0.95)]:
        sel = LAB_B[(LAB_B["t"] >= lo) & (LAB_B["t"] <= hi)]
        got = (sel["label"] == want).mean() if len(sel) else 0
        check(f"selftest broadcast {lo}-{hi}s labelled {want}", got >= share, f"{got:.0%} of {len(sel)}")
    for src in SOURCES:
        check(f"selftest {src} uses the fixed box", ROI_SOURCE[src] == "fixed" and ROI_RATE[src] >= ROI_MIN_RATE,
              f"{ROI[src]} reads {ROI_RATE[src]:.0%} of probes")
    lag = SYN_CLOCK_LAG_SEC        # the ATD refinement should recover it as +10 tactical frames
    for tb, tt in [(15.0, 20.24 + lag), (60.0, 70.9 + lag)]:
        row = BTAB[BTAB["broadcast_frame"] == int(round(tb * 25))].iloc[0]
        check(f"selftest map broadcast {tb}s -> tactical {tt:.2f}s (incl. ATD refinement)", abs(row["tactical_frame"] - tt * 25) <= 1,
              f"{row['tactical_frame']} vs {tt * 25:.1f}")
    for h in (1, 2):
        rep = REFINE_REPORT.get(h, {})
        check(f"selftest ATD refinement half {h} finds +10 frames", rep.get("status") == "accepted"
              and abs(rep.get("best_shift_frames", 99) - 10) <= 1, rep)
    check("selftest ATD orientation (half 1 identity, half 2 rot180)", VARIANT == {1: "identity", 2: "rot180"}, VARIANT)
    seek_df, _ = scan_video(VIDEO["tactical"], np.arange(300, 1300, 97), ROI["tactical"], mode="seek", desc="seek test")
    seq = SAMPLES["tactical"].set_index("frame")["clock"]
    exp = (np.floor(np.arange(300, 1300, 97) / 25 - 8.36 + 1e-9)).astype(float)
    exp[(np.arange(300, 1300, 97) / 25) >= 54.36] = np.nan
    agree = np.nanmean(np.where(np.isnan(exp), np.nan, seek_df["clock"].to_numpy() == exp))
    check("selftest seek-mode readings match the truth", agree >= 0.9, f"{agree:.0%}")

chk = pd.DataFrame(CHECKS)
print(chk.to_string(index=False))
n_fail = int((~chk["passed"] & chk["critical"]).sum())
print(f"\n{int(chk['passed'].sum())}/{len(chk)} passed, {n_fail} critical failure(s)")

## 15. Write the JSON (+ CSV) and validate it

Per-frame columns are stored column-wise (one list per field) with `start` / `stop` frames, labels as
integer codes, `-1` for "no frame" and `null` for NaN. That keeps the full-match file to a few MB.

In [ ]:
def _f(x, nd=3):
    x = float(x)
    return None if not math.isfinite(x) else round(x, nd)

def _segs_json(segs):
    return [{k: (_f(v) if isinstance(v, float) else v) for k, v in sg.items()} for sg in segs]

def source_block(s):
    blk = {"video": os.path.relpath(VIDEO[s], REPO), **INFO[s], "n_frames_expected": EXPECTED_FRAMES[s],
           "clock_roi_px": list(ROI[s]), "clock_roi_source": ROI_SOURCE[s], "sample_step": STEP[s], "n_samples": int(len(SAMPLES[s])),
           "ocr_read_rate": _f(np.isfinite(SAMPLES[s]["clock"]).mean()),
           "ocr_rejected_low_confidence": int(SAMPLES[s]["low_conf"].sum()) if "low_conf" in SAMPLES[s] else 0,
           "frozen_share": _f(SAMPLES[s]["frozen"].mean()),
           "segments": _segs_json(SEGS[s]),
           "kickoff": {str(h): {"frame": k["frame"], "time_sec": _f(k["time_sec"])} for h, k in KICKOFF[s].items()}}
    if s == "tactical":
        blk["kickoff_check"] = {str(h): {"expected": e, "measured": KICKOFF[s].get(h, {}).get("frame")}
                                for h, e in EXPECTED_TACTICAL_KICKOFF.items()}
    return blk

def col(a, nd=3):
    a = np.asarray(a)
    if a.dtype.kind == "f":
        return [None if not math.isfinite(v) else round(float(v), nd) for v in a]
    return a.astype(int).tolist()

OUT = {
    "schema_version": "1.0",
    "game": GAME,
    "run_mode": RUN_MODE,
    "created_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds"),
    "ocr": {"backend": OCR_BACKEND,
            "model": "RapidOCR PP-OCR recognition (ONNX), use_det=False" if OCR_BACKEND == "rapidocr" else OCR_BACKEND,
            "min_score": MIN_OCR_SCORE},
    "time_reference": {
        "description": ("match_clock_sec = what the match clock shows, in seconds (1st half starts at 0, 2nd at 2700, "
                        "stoppage time keeps counting); half_elapsed_sec = seconds since that half's kickoff. "
                        "video_time = frame / fps; within a segment match_clock = video_time - offset."),
        "half_start_clock_sec": {str(h): v for h, v in HALF_START_CLOCK.items()},
        "half_end_clock_sec": {str(h): _f(v) for h, v in END_CLOCK.items()},
    },
    "sources": {s: source_block(s) for s in SOURCES},
    "pair": {
        "description": "tactical_time = match_clock + tactical_offset(half) + extra_sec(half); broadcast likewise without extra",
        "extra_sec": {str(h): _f(v, 4) for h, v in PAIR_EXTRA.items()},
        "atd_orientation": {str(h): v for h, v in VARIANT.items()},
        "atd_refine": {str(h): v for h, v in REFINE_REPORT.items()},
    },
    "label_codes": {str(i): k for i, k in enumerate(LABELS)},
    "summary": {
        "broadcast_label_share": {k: _f(v) for k, v in BTAB["label"].value_counts(normalize=True).items()},
        "tactical_frames_with_live_broadcast": _f((TTAB["broadcast_frame"] >= 0).mean()),
        "runtime_sec": _f(time.time() - T0, 1),
    },
    "checks": CHECKS,
}
if WRITE_PER_FRAME:
    OUT["broadcast_frames"] = {
        "start": int(BTAB["broadcast_frame"].iloc[0]), "stop": int(BTAB["broadcast_frame"].iloc[-1]),
        "columns": {"half": col(BTAB["half"]), "match_clock_sec": col(BTAB["match_clock_sec"]),
                    "half_elapsed_sec": col(BTAB["half_elapsed_sec"]),
                    "label": col(BTAB["label"].map(LABEL_CODE)), "confidence": col(BTAB["confidence"], 2),
                    "tactical_frame": col(BTAB["tactical_frame"])}}
if WRITE_TACTICAL_INDEX:
    OUT["tactical_frames"] = {
        "start": int(TTAB["tactical_frame"].iloc[0]), "stop": int(TTAB["tactical_frame"].iloc[-1]),
        "columns": {"half": col(TTAB["half"]), "match_clock_sec": col(TTAB["match_clock_sec"]),
                    "broadcast_frame": col(TTAB["broadcast_frame"])}}

SCHEMA = {
    "type": "object",
    "required": ["schema_version", "game", "run_mode", "time_reference", "sources", "pair", "label_codes", "checks"],
    "properties": {
        "sources": {"type": "object", "required": list(SOURCES), "additionalProperties": {
            "type": "object", "required": ["fps", "n_frames", "segments", "kickoff"],
            "properties": {"segments": {"type": "array", "minItems": 1, "items": {
                "type": "object", "required": ["half", "t_first", "t_last", "offset", "precision_sec"]}}}}},
        "broadcast_frames": {"type": "object", "required": ["start", "stop", "columns"]},
        "tactical_frames": {"type": "object", "required": ["start", "stop", "columns"]},
        "checks": {"type": "array", "items": {"type": "object", "required": ["name", "passed"]}},
    },
}
try:
    import jsonschema
    jsonschema.validate(OUT, SCHEMA)
    for k in ("broadcast_frames", "tactical_frames"):
        if k in OUT:
            n = OUT[k]["stop"] - OUT[k]["start"] + 1
            assert all(len(v) == n for v in OUT[k]["columns"].values()), k
    check("output JSON valid against schema", True)
except ImportError:
    check("output JSON valid against schema", True, "jsonschema not installed: skipped", critical=False)
except Exception as exc:
    check("output JSON valid against schema", False, exc)
OUT["checks"] = CHECKS

json_path = OUT_DIR / f"sync_{RUN_MODE}.json"
with open(json_path, "w") as fh:
    json.dump(OUT, fh, separators=(",", ":"), allow_nan=False)
BTAB.to_csv(OUT_DIR / f"sync_{RUN_MODE}_broadcast_frames.csv", index=False)
print(f"wrote {json_path}  ({json_path.stat().st_size / 1e6:.1f} MB)")

## 16. Diagnostics

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 9))
for ax, s in zip(axes[:2], SOURCES):
    d = SAMPLES[s]; v = np.isfinite(d["clock"])
    ax.scatter(d["t"][v & ~d["frozen"]] / 60, (d["t"] - d["clock"])[v & ~d["frozen"]], s=2, label="reading")
    ax.scatter(d["t"][v & d["frozen"]] / 60, (d["t"] - d["clock"])[v & d["frozen"]], s=2, c="grey", label="frozen")
    for sg in SEGS[s]:
        ax.hlines(sg["offset"], sg["t_first"] / 60, sg["t_last"] / 60, colors="red", lw=2)
    for h, k in KICKOFF[s].items():
        ax.axvline(k["time_sec"] / 60, ls="--", c="k", lw=0.8)
    ax.set_title(f"{s}: video_time - clock (s); red = fitted segments, dashed = kickoffs"); ax.set_xlabel("video minute")
    ax.set_ylim(np.nanpercentile(d["t"] - d["clock"], 1) - 30, np.nanpercentile(d["t"] - d["clock"], 99) + 30); ax.legend(loc="upper left", fontsize=7)
colors = {"OFF_MATCH": "#bbbbbb", "LIVE_WIDE": "#2a9d4b", "LIVE_CLOSE": "#8fd19e", "REPLAY": "#d1495b", "UNKNOWN": "#f2c14e"}
ax = axes[2]
for k, c in colors.items():
    sel = LAB_B["label"] == k
    ax.scatter(LAB_B["t"][sel] / 60, np.zeros(sel.sum()), c=c, s=60, marker="|", label=k)
ax.set_yticks([]); ax.set_xlabel("broadcast video minute"); ax.set_title("broadcast sample labels"); ax.legend(ncol=5, fontsize=7)
plt.tight_layout(); plt.savefig(OUT_DIR / f"sync_{RUN_MODE}_offsets.png", dpi=110); plt.show()

if ATD_B is not None and np.isfinite(LAB_B["atd_m"]).any():
    plt.figure(figsize=(7, 3))
    for k in ("LIVE_WIDE", "REPLAY", "UNKNOWN"):
        x = LAB_B.loc[(LAB_B["label"] == k) & np.isfinite(LAB_B["atd_m"]), "atd_m"]
        if len(x):
            plt.hist(x.clip(upper=15), bins=60, alpha=0.5, label=f"{k} ({len(x)})", color=colors[k])
    plt.axvline(ATD_LIVE_M, c="k", ls="--"); plt.axvline(ATD_REPLAY_M, c="k", ls=":")
    plt.xlabel("ATD broadcast->tactical median nearest-player distance (m)"); plt.legend(); plt.tight_layout()
    plt.savefig(OUT_DIR / f"sync_{RUN_MODE}_labels.png", dpi=110); plt.show()

for s in SOURCES:
    if CROPS[s]:
        k = min(40, len(CROPS[s])); cols = 8; rows = int(math.ceil(k / cols))
        fig, axs = plt.subplots(rows, cols, figsize=(14, 1.3 * rows))
        for ax in np.atleast_1d(axs).ravel(): ax.axis("off")
        for ax, (f, crop, text) in zip(np.atleast_1d(axs).ravel(), sorted(CROPS[s])[:k]):
            ax.imshow(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)); ax.set_title(f"{f}: '{text}'", fontsize=6)
        plt.tight_layout(); plt.savefig(OUT_DIR / f"sync_{RUN_MODE}_ocr_audit_{s}.png", dpi=100); plt.close(fig)
print("diagnostics saved in", OUT_DIR)

## 17. Using the output elsewhere

`SyncMap` reads the JSON and answers both directions from the **segments** (works for any frame, even
outside the per-frame tables) and the per-frame labels when present.

In [ ]:
class SyncMap:
    def __init__(self, path):
        self.j = json.load(open(path))
        self.src = self.j["sources"]
        self.extra = {int(h): v for h, v in self.j["pair"]["extra_sec"].items() if v is not None}
        self.codes = {int(k): v for k, v in self.j["label_codes"].items()}
        self.tl = {s: Timeline(self.src[s]["segments"], self.src[s]["fps"], self.src[s]["n_frames"], s) for s in self.src}
        bf = self.j.get("broadcast_frames")
        self._lab = (bf["start"], np.asarray(bf["columns"]["label"])) if bf else None

    def broadcast_label(self, frame):
        if self._lab is None or not (self._lab[0] <= frame < self._lab[0] + len(self._lab[1])):
            return None
        return self.codes[int(self._lab[1][frame - self._lab[0]])]

    def match_time(self, source, frame):
        h, c = self.tl[source].to_match([frame / self.src[source]["fps"]])
        return int(h[0]), float(c[0])

    def broadcast_to_tactical(self, frame, live_only=True):
        if live_only and self.broadcast_label(frame) not in (None, "LIVE_WIDE", "LIVE_CLOSE"):
            return None
        h, c = self.match_time("broadcast", frame)
        t = self.tl["tactical"].from_match([h], [c], extra_offset=self.extra)[0]
        return None if not np.isfinite(t) else int(round(t * self.src["tactical"]["fps"]))

    def tactical_to_broadcast(self, frame, live_only=True):
        h, c = self.match_time("tactical", frame)
        t = self.tl["broadcast"].from_match([h], [c - self.extra.get(h, 0.0)])[0]
        if not np.isfinite(t):
            return None
        b = int(round(t * self.src["broadcast"]["fps"]))
        return None if live_only and self.broadcast_label(b) not in (None, "LIVE_WIDE", "LIVE_CLOSE") else b


sm = SyncMap(json_path)
for h, k in KICKOFF["tactical"].items():
    tb = sm.tactical_to_broadcast(k["frame"], live_only=False)
    print(f"half {h} kickoff: tactical frame {k['frame']} <-> broadcast frame {tb}  "
          f"(broadcast's own kickoff estimate {KICKOFF['broadcast'].get(h, {}).get('frame')})")
if RUN_MODE == "selftest":
    n_fail = int(sum(1 for c in CHECKS if c["critical"] and not c["passed"]))
    assert n_fail == 0, f"{n_fail} critical selftest check(s) failed"
    print("SELFTEST PASSED")

## 18. Reading the results

* Check §14 first: every critical row should be `True`. The tactical kickoff check compares against
  the documented frames 11,185 and 113,742. A miss there usually means a wrong ROI or a clock that
  isn't the match clock.
* `pair.atd_refine` shows how far the two clock graphics disagree (in frames) and whether the ATD
  refinement was accepted.
* `UNKNOWN` broadcast stretches are long clock-less shots with no homography and no slow-motion
  evidence (crowd, bench, studio). They're deliberately left unmapped.